# Practice 1 (v3): Map ADT and Hash Tables

## Learning objectives

- Use the main Map ADT operations and inspect its internal representation.
- Traverse a hash table containing buckets of key-value pairs.
- Explore collisions and understand separate chaining.
- Apply a map to frequency counting.
- Optionally, extend the ADT through inheritance.


## Instructions

This lab has a **guided exploration (5–10 min)**, **3 required challenges (45 min)**, and **an optional extension (15–20 min)**.

First, run the `Map` class **without changing it**. In Challenge 0, simply run the code and inspect the output. For later challenges, implement your solution in the indicated cell and run its `assert` tests afterward.

You may use AI tools to understand concepts or debug, but write your own solutions.


# 📚 Before starting: how does this hash table work?

This implementation uses a hash table with **separate chaining**:

- **Table and buckets:** `table` is a list of lists. Each inner list is a bucket containing zero or more key-value pairs.
- **Hash function:** `_hash(key)` converts a key into a valid index using modulo.
- **Collisions:** different keys may map to the same bucket.
- **Separate chaining:** each bucket stores multiple `[key, value]` pairs when collisions occur.
- **`put(key, value)`:** inserts a new pair or updates an existing key.
- **`get(key)`:** searches the relevant bucket and returns the value, or `None` if absent.
- **`remove(key)`:** removes the pair and returns its value, or `None` if absent.

> **Important:** map keys are **unique**, but values may repeat.


## Automated tests

Each challenge has a separate test cell using `assert`. When all assertions pass, you will see `Tests passed`. Otherwise, revise your implementation and run its cell again before rerunning the tests.


# 🧱 Starter code: `Map` class

This class implements a Map ADT using a hash table with fixed capacity `7` and separate chaining.

**Run this cell first and do not modify it.**


In [ ]:
class Map:
    """Map ADT implemented as a hash table with separate chaining."""

    def __init__(self):
        self._capacity = 7
        self.table = [[] for _ in range(self._capacity)]
        self.size = 0

    def _hash(self, key):
        """Internal hash function that returns a bucket index."""
        return abs(hash(key)) % self._capacity

    def put(self, key, value):
        """Insert a pair or update an existing key."""
        index = self._hash(key)
        bucket = self.table[index]

        # Update the value if the key already exists (unique keys).
        for item in bucket:
            if item[0] == key:
                item[1] = value
                return

        # Append [key, value] if this is a new key.
        bucket.append([key, value])
        self.size += 1

    def get(self, key):
        """Return the associated value, or None if absent."""
        index = self._hash(key)
        bucket = self.table[index]

        for item in bucket:
            if item[0] == key:
                return item[1]
        return None

    def remove(self, key):
        """Remove the key-value pair for this key."""
        index = self._hash(key)
        bucket = self.table[index]

        for i in range(len(bucket)):
            if bucket[i][0] == key:
                self.size -= 1
                return bucket.pop(i)[1]
        return None

    def __len__(self):
        return self.size

    def display(self):
        """Display the internal table structure."""
        print("\n--- Map Contents ---")
        for i in range(self._capacity):
            print(f"Index {i}: {self.table[i]}")
        print(f"Total elements: {len(self)}")
        print("--------------------\n")

# 🔎 Challenge 0. First steps with the Map ADT

🎯 **Goal:** observe what happens when inserting, looking up, updating, and removing associations.

🕒 **Time:** 5–10 minutes · 🟢 **Difficulty:** introductory.

**You do not need to implement any function.** Run the following cell and inspect the output of `display()`. Then answer the questions and experiment by adding two more people.


In [ ]:
# 1. Create an empty map
demo = Map()
demo.display()

# 2. Insert key-value pairs
demo.put("Ana", 20)
demo.put("Luis", 25)
demo.put("Marta", 22)
demo.display()

# 3. Look up a key
print("Ana's age:", demo.get("Ana"))

# 4. Update an existing key
demo.put("Ana", 21)
demo.display()

# 5. Remove a pair
demo.remove("Luis")
demo.display()


### 💭 Observe and experiment

1. What happens when you insert a key that already exists? Does the number of items change?
2. Why might associations appear in a different order from their insertion order?
3. How does `len(demo)` change when a key is removed?

**Try it:** add two people using `put()`, retrieve their ages using `get()`, and run `display()` to inspect their buckets. There are no automated tests for this introductory challenge.


> **Instructor notes:** `put()` replaces the value for an existing key without increasing `size`; bucket placement depends on `_hash()`, not insertion order; removing an existing key decreases the length by one.


# 🏆 Challenge 1. Retrieve all keys and values

🎯 **Goal:** traverse every bucket in the hash table.

🕒 **Time:** 15 minutes · 🟢 **Difficulty:** easy.

Implement `get_keys(map_obj)` and `get_values(map_obj)`. Each function should return a list of all keys or values, respectively, by traversing **every** bucket. Do not modify the map. Values may repeat; do not assume a particular iteration order.

> 💡 Hint: use one loop over buckets and another over the `[key, value]` pairs inside each bucket.


### 💻 Your solution


In [ ]:
def get_keys(map_obj: Map) -> list:
    """Return all keys without changing the map."""
    keys = []
    for bucket in map_obj.table:
        for key, value in bucket:
            keys.append(key)
    return keys


def get_values(map_obj: Map) -> list:
    """Return all values, including duplicates."""
    values = []
    for bucket in map_obj.table:
        for key, value in bucket:
            values.append(value)
    return values


### ✅ Automated tests


In [ ]:
m2 = Map()
m2.put(0, "Ana")
m2.put(7, "Carlos")  # Collides with 0: both bucket indices are 0.
m2.put(14, "Ana") # Three elements in the same bucket.
m2.put(1, "Lucía")

keys = get_keys(m2)
values = get_values(m2)
assert sorted(keys) == [0, 1, 7, 14]
assert sorted(values) == ["Ana", "Ana", "Carlos", "Lucía"]
assert len(keys) == len(values) == len(m2) == 4
assert len(m2.table[0]) == 3
assert m2.get(0) == "Ana" and m2.get(7) == "Carlos"
assert m2.get(14) == "Ana" and m2.get(1) == "Lucía"

empty = Map()
assert get_keys(empty) == [] and get_values(empty) == []
print("Tests passed")


# 🏆 Challenge 2. Explore collisions

🎯 **Goal:** understand what happens when two keys share a bucket.

🕒 **Time:** 10 minutes · 🟢 **Difficulty:** easy.

Implement `find_collision(map_obj)` to find **two distinct integers** from 0 through 99 with the same `_hash` index. Return `(key1, key2, index)`. Do not insert anything into the map inside this function.

The tests will insert both keys with `put`, check that they coexist in the same bucket, retrieve their values, and delete one without losing the other.

> 💡 Hint: store the first integer found for each index in an auxiliary dictionary. Alternatively, compare pairs of integers directly.


### 💻 Your solution


In [ ]:
def find_collision(map_obj: Map) -> tuple:
    """Find two different integers from 0..99 with the same bucket index."""
    first_seen = {}
    for key in range(100):
        index = map_obj._hash(key)
        if index in first_seen:
            return first_seen[index], key, index
        first_seen[index] = key
    raise ValueError("No collision found")


### ✅ Automated tests


In [ ]:
m1 = Map()
a, b, index = find_collision(m1)
assert isinstance(a, int) and isinstance(b, int) and a != b
assert 0 <= a < 100 and 0 <= b < 100
assert m1._hash(a) == m1._hash(b) == index
assert len(m1) == 0  # Finding a collision must not insert elements.

m1.put(a, "primero")
m1.put(b, "segundo")
assert len(m1) == 2
assert len(m1.table[index]) == 2
assert m1.get(a) == "primero" and m1.get(b) == "segundo"
assert m1.remove(a) == "primero"
assert m1.get(a) is None and m1.get(b) == "segundo"
assert len(m1) == 1 and len(m1.table[index]) == 1
print("Tests passed")


# 🏆 Challenge 3. Word frequency counter

🎯 **Goal:** apply `Map` to count words in text.

🕒 **Time:** 20 minutes · 🟡 **Difficulty:** intermediate.

Implement `WordCounter` containing `self.freq_map = Map()` and these methods:

- `add_text(text)`: lowercase the string, split it with `text.lower().split()`, and accumulate frequencies across multiple calls.
- `get_count(word)`: return the case-insensitive frequency, or `0` if the word is absent.
- `most_frequent()`: return `(word, count)` for a word with the highest frequency, or `(None, 0)` when empty. **Any tied word is acceptable.**

Do not strip punctuation: tokenization must use exactly `text.lower().split()`.

> 💡 Hint: read counts with `get()` and update with `put()`. To find the maximum, traverse every bucket.


### 💻 Your solution


In [ ]:
class WordCounter:
    def __init__(self):
        self.freq_map = Map()

    def add_text(self, text: str):
        for word in text.lower().split():
            current = self.freq_map.get(word)
            self.freq_map.put(word, 1 if current is None else current + 1)

    def get_count(self, word: str) -> int:
        result = self.freq_map.get(word.lower())
        return 0 if result is None else result

    def most_frequent(self) -> tuple:
        best_word, best_count = None, 0
        for bucket in self.freq_map.table:
            for word, count in bucket:
                if count > best_count:
                    best_word, best_count = word, count
        return best_word, best_count


### ✅ Automated tests


In [ ]:
wc = WordCounter()
assert wc.most_frequent() == (None, 0)
assert wc.get_count("hola") == 0
wc.add_text("Hola mundo hola python mundo hola")
assert wc.get_count("hola") == 3
assert wc.get_count("HOLA") == 3
assert wc.get_count("python") == 1
assert wc.get_count("java") == 0
assert wc.most_frequent() == ("hola", 3)

wc.add_text("MUNDO mundo")
assert wc.get_count("mundo") == 4
assert wc.most_frequent() == ("mundo", 4)
wc.add_text("  ")
assert wc.get_count("mundo") == 4

other = WordCounter()
other.add_text("uno dos")
word, count = other.most_frequent()
assert word in {"uno", "dos"} and count == 1
print("Tests passed")


# ⭐ Optional extension. Extend `Map` with `AdvancedMap`

🕒 **Time:** 15–20 minutes · 🟡 **Difficulty:** intermediate.

Create a subclass `AdvancedMap(Map)` with two methods:

- `clear()`: empty all buckets and reset `size` to zero.
- `keys_for_value(target_value)`: return all keys whose associated value matches the target, or `[]` if there are none.

This extension practices **inheritance and ADT operations**. It is not required to complete the three main challenges.


### 💻 Your solution


In [ ]:
class AdvancedMap(Map):
    def clear(self):
        self.table = [[] for _ in range(self._capacity)]
        self.size = 0

    def keys_for_value(self, target_value) -> list:
        matches = []
        for bucket in self.table:
            for key, value in bucket:
                if value == target_value:
                    matches.append(key)
        return matches


### ✅ Automated tests (optional)


In [ ]:
adv = AdvancedMap()
adv.put("A1", "Sistemas")
adv.put("A2", "Derecho")
adv.put("A3", "Sistemas")
assert sorted(adv.keys_for_value("Sistemas")) == ["A1", "A3"]
assert adv.keys_for_value("Medicina") == []
adv.clear()
assert len(adv) == 0
assert adv.get("A1") is None
assert all(bucket == [] for bucket in adv.table)
adv.put("B1", "Matemáticas")
assert len(adv) == 1 and adv.get("B1") == "Matemáticas"
print("Tests passed")


# 🎉 Practice complete!

After the required challenges, you will have explored collisions and separate chaining, traversed every bucket, and used a `Map` to count word frequencies. The optional extension also practices inheritance.
